In [1]:
# Make a CNN using CUDA Kernels only
"""
1 input
2 conv2d
3 maxPool
4 DenseRelu
5 DenseSigmoid
"""

'\n1 input\n2 conv2d\n3 maxPool\n4 DenseRelu\n5 DenseSigmoid\n\n'

In [2]:
import kagglehub
path = kagglehub.dataset_download("shaunthesheep/microsoft-catsvsdogs-dataset")

Using Colab cache for faster access to the 'microsoft-catsvsdogs-dataset' dataset.


In [3]:
print(path)

/kaggle/input/microsoft-catsvsdogs-dataset


In [6]:
import cv2
import numpy as np
import os

IMG_SIZE = 64

def load_data(data_path, limit=2000):
    images = []
    labels = []

    for label_name in ["Cat", "Dog"]:
        folder = os.path.join(data_path, "PetImages", label_name)
        label = 0 if label_name == "Cat" else 1

        count = 0
        for file in os.listdir(folder):
            if count >= limit:
                break
            try:
                img_path = os.path.join(folder, file)
                img = cv2.imread(img_path)
                img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
                img = img.astype(np.float32) / 255.0

                images.append(img)
                labels.append(label)
                count += 1
            except:
                continue

    return np.array(images), np.array(labels)

In [7]:
images, labels = load_data(path)
print(images.shape)

(4000, 64, 64)


In [8]:
from numba import cuda

In [9]:
@cuda.jit
def conv2d(input_img, kernel, output):
    h, w = cuda.grid(2)

    H = input_img.shape[0]
    W = input_img.shape[1]

    if h < H and w < W:
        sum_val = 0.0

        for kh in range(3):
            for kw in range(3):
                ih = h + kh - 1
                iw = w + kw - 1

                if 0 <= ih < H and 0 <= iw < W:
                    sum_val += input_img[ih, iw] * kernel[kh, kw]

        output[h, w] = sum_val

In [10]:
@cuda.jit
def relu(x):
    i, j = cuda.grid(2)

    if i < x.shape[0] and j < x.shape[1]:
        if x[i, j] < 0:
            x[i, j] = 0

In [11]:
@cuda.jit
def maxpool_kernel(input_img, output):
    h, w = cuda.grid(2)

    H_out = output.shape[0]
    W_out = output.shape[1]

    if h < H_out and w < W_out:
        max_val = -1e9

        for i in range(2):
            for j in range(2):
                val = input_img[h*2 + i, w*2 + j]
                if val > max_val:
                    max_val = val

        output[h, w] = max_val

In [12]:
@cuda.jit
def dense_forward(x, weights, bias, output):
    i = cuda.grid(1)

    if i < output.shape[0]:
        s = 0.0
        for j in range(x.shape[0]):
            s += x[j] * weights[j, i]
        output[i] = s + bias[i]

In [13]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

In [14]:
def binary_loss(pred, label):
    return -(label*np.log(pred+1e-8) +
             (1-label)*np.log(1-pred+1e-8))

In [15]:
def dense2_backward(hidden, pred, label, W2):
    dz = pred - label  # scalar

    grad_W2 = hidden.reshape(-1,1) * dz
    grad_b2 = dz

    grad_hidden = W2.flatten() * dz

    return grad_W2, grad_b2, grad_hidden

In [16]:
def dense2_backward(hidden, pred, label, W2):
    dz = pred - label  # scalar

    grad_W2 = hidden.reshape(-1,1) * dz
    grad_b2 = dz

    grad_hidden = W2.flatten() * dz

    return grad_W2, grad_b2, grad_hidden

In [17]:
def dense1_backward(flat, grad_hidden, W1):
    grad_W1 = np.outer(flat, grad_hidden)
    grad_b1 = grad_hidden
    grad_flat = np.dot(W1, grad_hidden)

    return grad_W1, grad_b1, grad_flat

In [18]:
def maxpool_forward_cpu(input_img):
    H, W = input_img.shape
    output = np.zeros((H//2, W//2), dtype=np.float32)
    mask = np.zeros_like(input_img, dtype=np.float32)

    for h in range(0, H, 2):
        for w in range(0, W, 2):
            patch = input_img[h:h+2, w:w+2]
            max_val = np.max(patch)
            output[h//2, w//2] = max_val

            for i in range(2):
                for j in range(2):
                    if patch[i,j] == max_val:
                        mask[h+i, w+j] = 1

    return output, mask

In [19]:
def maxpool_backward(grad_output, mask):
    H, W = mask.shape
    grad_input = np.zeros_like(mask)

    for h in range(grad_output.shape[0]):
        for w in range(grad_output.shape[1]):
            grad_input[h*2:h*2+2, w*2:w*2+2] += \
                mask[h*2:h*2+2, w*2:w*2+2] * grad_output[h,w]

    return grad_input

In [20]:
@cuda.jit
def conv2d_backward(input_img, grad_output, grad_kernel):
    kh, kw = cuda.grid(2)

    if kh < 3 and kw < 3:
        s = 0.0
        H = input_img.shape[0]
        W = input_img.shape[1]

        for h in range(H):
            for w in range(W):
                ih = h + kh - 1
                iw = w + kw - 1

                if 0 <= ih < H and 0 <= iw < W:
                    s += input_img[ih, iw] * grad_output[h, w]

        grad_kernel[kh, kw] = s

In [21]:
@cuda.jit
def conv2d_input_backward(grad_output, kernel, grad_input):
    h, w = cuda.grid(2)

    H = grad_input.shape[0]
    W = grad_input.shape[1]

    if h < H and w < W:
        s = 0.0

        for kh in range(3):
            for kw in range(3):
                oh = h - kh + 1
                ow = w - kw + 1

                if 0 <= oh < H and 0 <= ow < W:
                    s += grad_output[oh, ow] * kernel[kh, kw]

        grad_input[h, w] = s

In [23]:
@cuda.jit
def relu_backward_kernel(d_conv, d_grad):

    x, y = cuda.grid(2)

    if x < d_conv.shape[0] and y < d_conv.shape[1]:

        if d_conv[x, y] <= 0:
            d_grad[x, y] = 0.0

In [29]:
# Dense layer sizes
FLAT_SIZE = 32 * 32
HIDDEN = 64

W1 = np.random.randn(FLAT_SIZE, HIDDEN).astype(np.float32) * 0.01
b1 = np.zeros(HIDDEN, dtype=np.float32)

W2 = np.random.randn(HIDDEN, 1).astype(np.float32) * 0.01
b2 = np.zeros(1, dtype=np.float32)

In [33]:
import math

def launch_2d(kernel, *args):

    threads_per_block = (16, 16)

    # assume first argument is output array
    out = args[-1]

    blocks_x = math.ceil(out.shape[0] / threads_per_block[0])
    blocks_y = math.ceil(out.shape[1] / threads_per_block[1])

    blocks_per_grid = (blocks_x, blocks_y)

    kernel[blocks_per_grid, threads_per_block](*args)

In [ ]:
KERNEL_SIZE = 3

# Xavier-style small random init
conv_kernel = np.random.randn(KERNEL_SIZE, KERNEL_SIZE).astype(np.float32) * 0.01

# Move to GPU
d_conv_kernel = cuda.to_device(conv_kernel)

In [34]:
LR = 0.01

# Initialize conv_kernel (moved from a separate cell to ensure definition)
KERNEL_SIZE = 3
conv_kernel = np.random.randn(KERNEL_SIZE, KERNEL_SIZE).astype(np.float32) * 0.01
d_conv_kernel = cuda.to_device(conv_kernel)

for epoch in range(5):

    total_loss = 0

    for i in range(len(images)):

        img = images[i]
        label = labels[i]

        # ---------------- GPU Forward ----------------

        d_img = cuda.to_device(img)

        d_conv = cuda.device_array((IMG_SIZE, IMG_SIZE), dtype=np.float32)
        launch_2d(conv2d, d_img, d_conv_kernel, d_conv)

        launch_2d(relu, d_conv)

        d_pool = cuda.device_array((32,32), dtype=np.float32)
        maxpool_kernel[(2,2),(16,16)](d_conv, d_pool)

        # ------------- Flatten to CPU ---------------

        pool_out = d_pool.copy_to_host()
        flat = pool_out.flatten()

        # ---------------- Dense Forward --------------

        hidden = np.dot(flat, W1) + b1
        hidden_relu = np.maximum(hidden, 0)

        out = np.dot(hidden_relu, W2) + b2
        pred = sigmoid(out)

        loss = binary_loss(pred, label)
        total_loss += loss

        # ---------------- Backprop -------------------

        # dLoss/dOut
        d_out = pred - label

        # Dense2 gradients
        dW2 = np.outer(hidden_relu, d_out)
        db2 = d_out

        # Backprop to hidden
        d_hidden = np.dot(W2, d_out)
        d_hidden[hidden <= 0] = 0

        # Dense1 gradients
        dW1 = np.outer(flat, d_hidden)
        db1 = d_hidden

        # Update Dense Weights
        W2 -= LR * dW2
        b2 -= LR * db2
        W1 -= LR * dW1
        b1 -= LR * db1

        # -------- Backprop to Pool Layer ------------

        d_flat = np.dot(W1, d_hidden)
        d_pool_host = d_flat.reshape(32,32)

        # Send gradient to GPU
        d_pool_grad = cuda.to_device(d_pool_host)

        # Allocate grad for conv output
        d_conv_grad = cuda.device_array((IMG_SIZE, IMG_SIZE), dtype=np.float32)

        # Backprop through MaxPool
        conv_host = d_conv.copy_to_host()

        pool_out, pool_mask = maxpool_forward_cpu(conv_host)


        # Backprop through ReLU
        relu_backward_kernel[(IMG_SIZE, IMG_SIZE),1](d_conv, d_conv_grad)

        # Backprop through Conv
        d_kernel_grad = cuda.device_array(d_conv_kernel.shape, dtype=np.float32)
        conv2d_backward[(IMG_SIZE, IMG_SIZE),1](d_img, d_conv_grad, d_kernel_grad)

        # Update Conv Kernel
        kernel_grad_host = d_kernel_grad.copy_to_host()
        d_conv_kernel = cuda.to_device(
            d_conv_kernel.copy_to_host() - LR * kernel_grad_host
        )

    print("Epoch", epoch, "Loss:", total_loss/len(images))

/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 16 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 16 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


Epoch 0 Loss: [0.30881566]
Epoch 1 Loss: [0.42933173]
Epoch 2 Loss: [0.42997258]
Epoch 3 Loss: [0.42997536]
Epoch 4 Loss: [0.42997536]


In [35]:
def predict(img):

    # Move image to GPU
    d_img = cuda.to_device(img)

    # ----- Conv -----
    d_conv = cuda.device_array((IMG_SIZE, IMG_SIZE), dtype=np.float32)
    launch_2d(conv2d, d_img, d_conv_kernel, d_conv)
    launch_2d(relu, d_conv)

    # ----- Pool -----
    d_pool = cuda.device_array((32,32), dtype=np.float32)
    maxpool_kernel[(2,2),(16,16)](d_conv, d_pool)

    # Flatten
    flat = d_pool.copy_to_host().flatten()

    # ----- Dense -----
    hidden = np.dot(flat, W1) + b1
    hidden = np.maximum(hidden, 0)

    out = np.dot(hidden, W2) + b2
    prob = sigmoid(out)

    # Binary classification
    prediction = 1 if prob > 0.5 else 0

    return prediction, float(prob)

In [36]:
test_img = images[0]

pred, prob = predict(test_img)

print("Prediction:", pred)
print("Probability:", prob)
print("Actual Label:", labels[0])

Prediction: 1
Probability: 0.9363261461257935
Actual Label: 0


/tmp/ipykernel_849/2747106200.py:28: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return prediction, float(prob)
